# 국민여행조사 시군구별 전체여행 특성(2023~2025년)

이 노트북은 2023~2025년 국민여행조사 통합 전처리본에서 229개
시군구의 wide 특성표를 다시 계산한다. 공통 여행문항은 CASE 1~5
모든 여행에 `WT_DOM × 시군구 체류일 비중`을 적용하고, A시리즈는
`CHECK='Y'`인 대표여행 중 CASE 1·2·4에만 `WT_DOM`을 적용한다.

CASE 1–CASE 2 비용 격차는 산출하지 않는다. A시리즈 특성은 전체여행
지표가 아니라 대표여행 응답 특성으로만 해석한다.

In [ ]:
from pathlib import Path
import sys
import warnings

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "path.py").is_file():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise FileNotFoundError("프로젝트 루트를 찾지 못했습니다.")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from src.path import (
    NATIONAL_TRAVEL_SURVEY_PREPROCESSED_DATA_DIR,
    NATIONAL_TRAVEL_SURVEY_RAW_DATA_DIR,
    OUTPUTS_DIR,
    PREPROCESS_DATA_DIR,
    create_output_directories,
)
from src.visualization import COLORS, apply_plot_style

warnings.filterwarnings("ignore", category=FutureWarning)
apply_plot_style()

NOTEBOOK_STEM = "260726_국민여행조사_시군구별_전체여행_특성_3개년"
FIGURES_DIR, TABLES_DIR = create_output_directories(f"{NOTEBOOK_STEM}.ipynb")
FEATURE_DIR = PREPROCESS_DATA_DIR / "features_by_region"
FEATURE_PATH = FEATURE_DIR / "sigungu_all_trip_features_2023_2025.csv"
FEATURE_CODEBOOK_PATH = (
    FEATURE_DIR / "sigungu_all_trip_features_2023_2025_codebook.csv"
)
DATA_PATH = (
    NATIONAL_TRAVEL_SURVEY_PREPROCESSED_DATA_DIR
    / "national_travel_survey_2023_2025_all_tour.csv"
)
CODEBOOK_PATH = (
    NATIONAL_TRAVEL_SURVEY_PREPROCESSED_DATA_DIR
    / "national_travel_survey_2023_2025_all_tour_codebook.csv"
)
REGION_CODE_PATH = (
    NATIONAL_TRAVEL_SURVEY_RAW_DATA_DIR
    / "national_travel_survey_region_code.csv"
)
LEGACY_FEATURE_PATH = (
    OUTPUTS_DIR
    / "260720_국민여행조사_지역별_전체여행_집계_3개년"
    / "tables"
    / "region_trip_feature.csv"
)
CLUSTER_LABEL_PATH = (
    OUTPUTS_DIR
    / "260720_연고관광_사전층화_클러스터링_3개년_v4"
    / "tables"
    / "baseline_stratified_cluster_labeled_data.csv"
)
TRIP_SLOTS = range(1, 7)
MAX_SPOT = 17
CASE_LABELS = {
    1: "CASE1_관광휴양",
    2: "CASE2_연고방문_관광포함",
    3: "CASE3_연고방문_관광미포함",
    4: "CASE4_출장업무_관광포함",
    5: "CASE5_출장업무_관광미포함",
}
CLUSTER_LABELS = [
    "당일여행 C1",
    "당일여행 C2",
    "숙박여행 C1",
    "숙박여행 C2",
    "숙박여행 C3",
]
A_COLUMNS = ["A2_11", "A4_1", "A4_2", "A4_6", "A4_18", "A9", "A10"]

print(f"노트북: notebooks/05_Features_by_Region/{NOTEBOOK_STEM}.ipynb")
print(f"최종 특성표: {FEATURE_PATH.relative_to(PROJECT_ROOT)}")
print(f"최종 코드북: {FEATURE_CODEBOOK_PATH.relative_to(PROJECT_ROOT)}")
print(f"실행별 QC: {TABLES_DIR.parent.relative_to(PROJECT_ROOT)}")

## 1. 분석 변수 계약

분석 단위와 최종 열 순서를 먼저 선언한다. 여행-시군구 자료에서는 같은
여행이 같은 시군구를 여러 번 방문한 경우 체류일을 합산해 한 행으로
축약한다. 대표여행 A시리즈는 이 축약된 방문 시군구마다 한 번씩만
배정하며 체류일 비중으로 나누지 않는다.

In [ ]:
def trip_columns() -> list[str]:
    '''필요한 여행·방문지 원천 변수 목록을 만든다.

    Returns:
        all_tour CSV에서 읽을 열 이름 목록.
    '''
    columns = ["YEAR", "ID", "WT_DOM", *A_COLUMNS]
    for trip_no in TRIP_SLOTS:
        columns.extend(
            [
                f"D_TRA{trip_no}_CASE",
                f"D_TRA{trip_no}_CHECK",
                f"D_TRA{trip_no}_S_Day",
                f"D_TRA{trip_no}_COST",
                f"D_TRA{trip_no}_NUM",
                f"D_TRA{trip_no}_ONE_COST",
            ]
        )
    return columns


def spot_columns() -> list[str]:
    '''방문지 변환에만 필요한 원천 변수 목록을 만든다.

    Returns:
        청크 단위로 읽을 방문지 관련 열 이름 목록.
    '''
    columns = ["YEAR", "ID"]
    for trip_no in TRIP_SLOTS:
        for spot_no in range(1, MAX_SPOT + 1):
            prefix = f"D_TRA{trip_no}_{spot_no}_"
            columns.extend(
                [
                    f"{prefix}SPOT",
                    f"{prefix}SYEAR",
                    f"{prefix}SMONTH",
                    f"{prefix}SDAY",
                    f"{prefix}EYEAR",
                    f"{prefix}EMONTH",
                    f"{prefix}EDAY",
                    f"{prefix}Q6",
                ]
            )
    return columns


OUTPUT_COLUMNS = [
    "spot_code", "시도명", "시군구명", "비가중_여행수", "가중_지역규모",
    "전국_대비_비중(%)", *CASE_LABELS.values(),
    *[f"case{case}_trip_n_all_trip_case15" for case in CASE_LABELS],
    *[f"case{case}_share_weighted_all_trip_case15_pct" for case in CASE_LABELS],
    "hometown_visit_n_all_trip_case23",
    "hometown_visit_weighted_all_trip_case23",
    "hometown_visit_share_w_all_trip_case15_pct",
    "rank_by_hometown_visit_weighted_all_trip_case23",
    "rank_by_hometown_visit_share_all_trip_case23",
    *[f"군집{label}_가중규모" for label in CLUSTER_LABELS],
    "연고관광_가중규모_합",
    *[f"군집{label}_구성비" for label in CLUSTER_LABELS],
    "total_cost_mean_w_all_trip_case15",
    "total_cost_median_w_all_trip_case15",
    "one_person_cost_mean_w_all_trip_case15",
    "one_person_cost_median_w_all_trip_case15",
    "travel_nights_mean_w_all_trip_case15",
    "overnight_rate_w_all_trip_case15_pct",
    "family_friend_lodging_rate_w_all_trip_case15_pct",
    "rep_trip_n_case124",
    "rep_trip_weighted_denom_case124",
    "family_friend_lodging_rate_w_rep_trip_case124_pct",
    "no_reservation_rate_w_rep_trip_case124_pct",
    "nature_activity_rate_w_rep_trip_case124_pct",
    "food_activity_rate_w_rep_trip_case124_pct",
    "rest_activity_rate_w_rep_trip_case124_pct",
    "family_visit_activity_rate_w_rep_trip_case124_pct",
    "satisfaction_mean_w_rep_trip_case124",
    "revisit_intent_mean_w_rep_trip_case124",
    "activity_potential_rep_trip_case124_pct",
    "low_spending_conversion_hint_rep_trip_case124",
    "is_reliable_rep_trip_case124",
]
assert len(OUTPUT_COLUMNS) == len(set(OUTPUT_COLUMNS))

## 2. 원천 자료·코드북·지역코드 로드

In [ ]:
codebook = pd.read_csv(CODEBOOK_PATH, encoding="utf-8-sig")
required_columns = trip_columns()
required_codebook_columns = required_columns + spot_columns()
codebook_columns = set(codebook["column_name"])
missing_codebook_columns = sorted(
    set(required_codebook_columns) - codebook_columns
)
assert not missing_codebook_columns, (
    f"통합 코드북에 없는 필수 열: {missing_codebook_columns}"
)

data = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig",
    usecols=required_columns,
    low_memory=False,
)
region_codes = pd.read_csv(REGION_CODE_PATH, encoding="utf-8-sig")
region_codes["spot_code"] = (
    region_codes["시도_코드"].astype(int) * 1000
    + region_codes["시군구_코드"].astype(int)
)
assert len(region_codes) == 229
assert region_codes["spot_code"].is_unique
assert not region_codes.duplicated(["시도명", "시군구명"]).any()

legacy_feature = pd.read_csv(LEGACY_FEATURE_PATH, encoding="utf-8-sig")
cluster_labels = pd.read_csv(
    CLUSTER_LABEL_PATH,
    encoding="utf-8-sig",
    usecols=["YEAR", "ID", "군집"],
)
assert not cluster_labels.duplicated(["YEAR", "ID"]).any()
print(f"원천 자료: {data.shape[0]:,}행 × {data.shape[1]:,}열")
print(f"기존 회귀 기준: {legacy_feature.shape[0]}행")
print(f"군집 라벨: {cluster_labels.shape[0]:,}행")

## 3. 키·가중치·여행 슬롯 품질검사와 여행 long 변환

In [ ]:
assert data["ID"].notna().all()
assert not data.duplicated(["YEAR", "ID"]).any()
data["WT_DOM"] = pd.to_numeric(data["WT_DOM"], errors="coerce")
assert data["WT_DOM"].notna().all() and data["WT_DOM"].gt(0).all()

trip_frames = []
for trip_no in TRIP_SLOTS:
    case = pd.to_numeric(data[f"D_TRA{trip_no}_CASE"], errors="coerce")
    valid = case.notna()
    trip_frames.append(
        pd.DataFrame(
            {
                "YEAR": data.loc[valid, "YEAR"].to_numpy(),
                "ID": data.loc[valid, "ID"].to_numpy(),
                "TRIP_NO": trip_no,
                "WT_DOM": data.loc[valid, "WT_DOM"].to_numpy(),
                "CASE": case.loc[valid].astype(int).to_numpy(),
                "CHECK": data.loc[valid, f"D_TRA{trip_no}_CHECK"].to_numpy(),
                "travel_nights": pd.to_numeric(
                    data.loc[valid, f"D_TRA{trip_no}_S_Day"], errors="coerce"
                ).to_numpy(),
                "total_cost": pd.to_numeric(
                    data.loc[valid, f"D_TRA{trip_no}_COST"], errors="coerce"
                ).to_numpy(),
                "one_person_cost": pd.to_numeric(
                    data.loc[valid, f"D_TRA{trip_no}_ONE_COST"], errors="coerce"
                ).to_numpy(),
            }
        )
    )
trip_long = pd.concat(trip_frames, ignore_index=True)
assert not trip_long.duplicated(["YEAR", "ID", "TRIP_NO"]).any()
assert trip_long["CASE"].between(1, 5).all()
expected_case_n = {1: 59514, 2: 13107, 3: 8638, 4: 1518, 5: 1896}
actual_case_n = trip_long["CASE"].value_counts().sort_index().to_dict()
assert actual_case_n == expected_case_n, actual_case_n
assert trip_long[["travel_nights", "total_cost", "one_person_cost"]].notna().all().all()
assert trip_long[["total_cost", "one_person_cost"]].ge(0).all().all()
assert trip_long["travel_nights"].ge(0).all()
assert data[[f"D_TRA{trip_no}_CHECK" for trip_no in TRIP_SLOTS]].eq("Y").sum(axis=1).le(1).all()
display(trip_long["CASE"].value_counts().sort_index().to_frame("여행 수"))

## 4. 방문지 long 변환, 체류일·숙박 정보 계산, 시군구 중복 제거

In [ ]:
def region_days_from_dates(frame: pd.DataFrame, prefix: str) -> tuple[pd.Series, pd.Series]:
    '''방문 시작·종료일에서 체류일과 파싱 실패 여부를 계산한다.

    Args:
        frame: 방문지 슬롯 원천 자료.
        prefix: 해당 방문지 슬롯 열 접두어.

    Returns:
        유효 체류일과 날짜 파싱 실패 여부의 튜플.
    '''
    start = pd.to_datetime(
        {
            "year": pd.to_numeric(frame[f"{prefix}SYEAR"], errors="coerce"),
            "month": pd.to_numeric(frame[f"{prefix}SMONTH"], errors="coerce"),
            "day": pd.to_numeric(frame[f"{prefix}SDAY"], errors="coerce"),
        },
        errors="coerce",
    )
    end = pd.to_datetime(
        {
            "year": pd.to_numeric(frame[f"{prefix}EYEAR"], errors="coerce"),
            "month": pd.to_numeric(frame[f"{prefix}EMONTH"], errors="coerce"),
            "day": pd.to_numeric(frame[f"{prefix}EDAY"], errors="coerce"),
        },
        errors="coerce",
    )
    parse_failed = start.isna() | end.isna()
    days = (end - start).dt.days + 1
    return days.where(~parse_failed & days.gt(0)), parse_failed


spot_frames = []
spot_source = pd.read_csv(
    DATA_PATH,
    encoding="utf-8-sig",
    usecols=spot_columns(),
    engine="pyarrow",
)
for source_chunk in [spot_source]:
    for trip_no in TRIP_SLOTS:
        for spot_no in range(1, MAX_SPOT + 1):
            prefix = f"D_TRA{trip_no}_{spot_no}_"
            spot_code = pd.to_numeric(source_chunk[f"{prefix}SPOT"], errors="coerce")
            valid = spot_code.notna()
            if not valid.any():
                continue
            slot_columns = [
                "YEAR",
                "ID",
                f"{prefix}SYEAR",
                f"{prefix}SMONTH",
                f"{prefix}SDAY",
                f"{prefix}EYEAR",
                f"{prefix}EMONTH",
                f"{prefix}EDAY",
                f"{prefix}Q6",
            ]
            slot_data = source_chunk.loc[valid, slot_columns]
            days, parse_failed = region_days_from_dates(slot_data, prefix)
            spot_frames.append(
                pd.DataFrame(
                    {
                        "YEAR": slot_data["YEAR"].to_numpy(),
                        "ID": slot_data["ID"].to_numpy(),
                        "TRIP_NO": trip_no,
                        "SPOT_NO": spot_no,
                        "spot_code": spot_code.loc[valid].astype(int).to_numpy(),
                        "region_days": days.to_numpy(),
                        "date_parse_failed": parse_failed.to_numpy(),
                        "Q6": pd.to_numeric(slot_data[f"{prefix}Q6"], errors="coerce").to_numpy(),
                    }
                )
            )
spot_long = pd.concat(spot_frames, ignore_index=True)
spot_long = spot_long.merge(
    region_codes[["spot_code", "시도명", "시군구명"]],
    on="spot_code",
    how="left",
    validate="many_to_one",
)
unmatched_spot_codes = (
    spot_long.loc[spot_long["시군구명"].isna()]
    .groupby("spot_code").size().rename("발생_건수").reset_index()
)
valid_spots = spot_long.loc[
    spot_long["region_days"].notna() & spot_long["시군구명"].notna()
].copy()
dedup_before = len(valid_spots)
trip_region_days = (
    valid_spots.groupby(["YEAR", "ID", "TRIP_NO", "spot_code"], as_index=False)
    .agg(region_days=("region_days", "sum"), family_friend_lodging=("Q6", lambda value: value.eq(12).any()))
)
trip_region_days["trip_day_sum"] = trip_region_days.groupby(
    ["YEAR", "ID", "TRIP_NO"]
)["region_days"].transform("sum")
trip_region_days["day_share"] = (
    trip_region_days["region_days"] / trip_region_days["trip_day_sum"]
)
trip_region = trip_region_days.merge(
    trip_long,
    on=["YEAR", "ID", "TRIP_NO"],
    how="left",
    validate="many_to_one",
).merge(
    region_codes[["spot_code", "시도명", "시군구명"]],
    on="spot_code",
    how="left",
    validate="many_to_one",
)
trip_region["weighted_region_size"] = trip_region["WT_DOM"] * trip_region["day_share"]
assert trip_region["시군구명"].notna().all()
assert np.allclose(
    trip_region.groupby(["YEAR", "ID", "TRIP_NO"])["day_share"].sum(), 1.0, atol=1e-6
)
qc_visit = pd.DataFrame(
    [
        {"검증항목": "방문지 슬롯 원본 행 수", "값": len(spot_long)},
        {"검증항목": "날짜 파싱 실패 행 수", "값": int(spot_long["date_parse_failed"].sum())},
        {"검증항목": "지역코드 미매칭 행 수", "값": int(spot_long["시군구명"].isna().sum())},
        {"검증항목": "유효 방문지 dedup 전 행 수", "값": dedup_before},
        {"검증항목": "여행-시군구 dedup 후 행 수", "값": len(trip_region)},
    ]
)
display(qc_visit)

## 5. 전체여행 규모·CASE 구성·CASE 2+3 연고방문 지표

In [ ]:
keys = ["spot_code", "시도명", "시군구명"]
all_region = region_codes[["spot_code", "시도명", "시군구명"]].copy()
scale = (
    trip_region.groupby(keys, as_index=False)
    .agg(비가중_여행수=("TRIP_NO", "size"), 가중_지역규모=("weighted_region_size", "sum"))
)
feature = all_region.merge(scale, on=keys, how="left")
feature[["비가중_여행수", "가중_지역규모"]] = feature[
    ["비가중_여행수", "가중_지역규모"]
].fillna(0.0)
feature["전국_대비_비중(%)"] = feature["가중_지역규모"] / feature["가중_지역규모"].sum() * 100

case_weight = (
    trip_region.assign(case_column=trip_region["CASE"].map(CASE_LABELS))
    .pivot_table(index=keys, columns="case_column", values="weighted_region_size", aggfunc="sum", fill_value=0.0)
    .reindex(columns=list(CASE_LABELS.values()), fill_value=0.0).reset_index()
)
case_n = (
    trip_region.pivot_table(index=keys, columns="CASE", values="TRIP_NO", aggfunc="size", fill_value=0)
    .reindex(columns=list(CASE_LABELS), fill_value=0).reset_index()
)
case_n.columns = keys + [f"case{case}_trip_n_all_trip_case15" for case in CASE_LABELS]
feature = feature.merge(case_weight, on=keys, how="left").merge(case_n, on=keys, how="left")
for case, column in CASE_LABELS.items():
    feature[column] = feature[column].fillna(0.0)
    feature[f"case{case}_trip_n_all_trip_case15"] = feature[
        f"case{case}_trip_n_all_trip_case15"
    ].fillna(0).astype(int)
    feature[f"case{case}_share_weighted_all_trip_case15_pct"] = np.where(
        feature["가중_지역규모"].gt(0), feature[column] / feature["가중_지역규모"] * 100, np.nan
    )
assert np.allclose(feature[list(CASE_LABELS.values())].sum(axis=1), feature["가중_지역규모"], atol=1e-6)
assert (feature[[f"case{case}_trip_n_all_trip_case15" for case in CASE_LABELS]].sum(axis=1) == feature["비가중_여행수"]).all()

hometown = trip_region.loc[trip_region["CASE"].isin([2, 3])].groupby(keys, as_index=False).agg(
    hometown_visit_n_all_trip_case23=("TRIP_NO", "size"),
    hometown_visit_weighted_all_trip_case23=("weighted_region_size", "sum"),
)
feature = feature.merge(hometown, on=keys, how="left")
feature[["hometown_visit_n_all_trip_case23", "hometown_visit_weighted_all_trip_case23"]] = feature[
    ["hometown_visit_n_all_trip_case23", "hometown_visit_weighted_all_trip_case23"]
].fillna(0.0)
feature["hometown_visit_n_all_trip_case23"] = feature["hometown_visit_n_all_trip_case23"].astype(int)
feature["hometown_visit_share_w_all_trip_case15_pct"] = np.where(
    feature["가중_지역규모"].gt(0),
    feature["hometown_visit_weighted_all_trip_case23"] / feature["가중_지역규모"] * 100,
    np.nan,
)
feature["rank_by_hometown_visit_weighted_all_trip_case23"] = feature[
    "hometown_visit_weighted_all_trip_case23"
].rank(method="min", ascending=False).astype(int)
feature["rank_by_hometown_visit_share_all_trip_case23"] = feature[
    "hometown_visit_share_w_all_trip_case15_pct"
].rank(method="min", ascending=False, na_option="bottom").astype(int)

## 6. CASE 1~5 공통 비용·기간·숙박 특성

In [ ]:
def weighted_median(values: pd.Series, weights: pd.Series) -> float:
    '''유효 관측치의 가중 중앙값을 계산한다.

    Args:
        values: 중앙값을 구할 수치형 값.
        weights: values와 같은 순서의 양수 가중치.

    Returns:
        누적 가중치가 50%에 처음 도달하는 값 또는 결측값.
    '''
    paired = pd.DataFrame({"value": values, "weight": weights}).dropna()
    paired = paired.loc[paired["weight"].gt(0)].sort_values("value")
    if paired.empty:
        return float("nan")
    return float(paired.loc[paired["weight"].cumsum().ge(paired["weight"].sum() / 2), "value"].iloc[0])


def weighted_mean(values: pd.Series, weights: pd.Series) -> float:
    '''결측을 쌍별 제외한 가중 평균을 계산한다.

    Args:
        values: 평균을 구할 수치형 값.
        weights: values와 같은 순서의 양수 가중치.

    Returns:
        가중 평균 또는 유효값이 없을 때 결측값.
    '''
    valid = values.notna() & weights.notna() & weights.gt(0)
    return float(np.average(values.loc[valid], weights=weights.loc[valid])) if valid.any() else float("nan")


common_rows = []
for spot_code, subset in trip_region.groupby("spot_code"):
    weight = subset["weighted_region_size"]
    common_rows.append(
        {
            "spot_code": spot_code,
            "total_cost_mean_w_all_trip_case15": weighted_mean(subset["total_cost"], weight),
            "total_cost_median_w_all_trip_case15": weighted_median(subset["total_cost"], weight),
            "one_person_cost_mean_w_all_trip_case15": weighted_mean(subset["one_person_cost"], weight),
            "one_person_cost_median_w_all_trip_case15": weighted_median(subset["one_person_cost"], weight),
            "travel_nights_mean_w_all_trip_case15": weighted_mean(subset["travel_nights"], weight),
            "overnight_rate_w_all_trip_case15_pct": weighted_mean(subset["travel_nights"].ge(1).astype(float), weight) * 100,
            "family_friend_lodging_rate_w_all_trip_case15_pct": weighted_mean(subset["family_friend_lodging"].astype(float), weight) * 100,
        }
    )
common_feature = pd.DataFrame(common_rows)
feature = feature.merge(common_feature, on="spot_code", how="left", validate="one_to_one")

## 7. 대표여행 CASE 1·2·4 A시리즈 특성과 정책점수

In [ ]:
representative_trip = trip_long.loc[trip_long["CHECK"].eq("Y")].copy()
assert len(representative_trip) == 72133
assert representative_trip["CASE"].value_counts().sort_index().to_dict() == {1: 57904, 2: 12790, 4: 1439}
assert representative_trip["CASE"].isin([1, 2, 4]).all()

a_data = data[["YEAR", "ID", *A_COLUMNS]].copy()
rep_region = trip_region.merge(
    representative_trip[["YEAR", "ID", "TRIP_NO"]],
    on=["YEAR", "ID", "TRIP_NO"],
    how="inner",
    validate="many_to_one",
).merge(a_data, on=["YEAR", "ID"], how="left", validate="many_to_one")
assert not rep_region.duplicated(["YEAR", "ID", "TRIP_NO", "spot_code"]).any()
for column in ["A2_11", "A4_1", "A4_2", "A4_6", "A4_18"]:
    rep_region[column] = pd.to_numeric(rep_region[column], errors="coerce")
    assert rep_region[column].notna().all() and rep_region[column].isin([0, 1]).all(), column
for column in ["A9", "A10"]:
    rep_region[column] = pd.to_numeric(rep_region[column], errors="coerce")
    assert rep_region[column].between(1, 5).all(), column

rep_rows = []
for spot_code, subset in rep_region.groupby("spot_code"):
    weight = subset["WT_DOM"]
    measures = {
        "family_friend_lodging_rate_w_rep_trip_case124_pct": weighted_mean(subset["family_friend_lodging"].astype(float), weight) * 100,
        "no_reservation_rate_w_rep_trip_case124_pct": weighted_mean(subset["A2_11"], weight) * 100,
        "nature_activity_rate_w_rep_trip_case124_pct": weighted_mean(subset["A4_1"], weight) * 100,
        "food_activity_rate_w_rep_trip_case124_pct": weighted_mean(subset["A4_2"], weight) * 100,
        "rest_activity_rate_w_rep_trip_case124_pct": weighted_mean(subset["A4_6"], weight) * 100,
        "family_visit_activity_rate_w_rep_trip_case124_pct": weighted_mean(subset["A4_18"], weight) * 100,
        "satisfaction_mean_w_rep_trip_case124": weighted_mean(subset["A9"], weight),
        "revisit_intent_mean_w_rep_trip_case124": weighted_mean(subset["A10"], weight),
    }
    measures["activity_potential_rep_trip_case124_pct"] = np.mean(
        [measures["nature_activity_rate_w_rep_trip_case124_pct"], measures["food_activity_rate_w_rep_trip_case124_pct"], measures["rest_activity_rate_w_rep_trip_case124_pct"]]
    )
    measures["low_spending_conversion_hint_rep_trip_case124"] = (
        0.35 * measures["family_friend_lodging_rate_w_rep_trip_case124_pct"]
        + 0.25 * measures["no_reservation_rate_w_rep_trip_case124_pct"]
        + 0.25 * measures["activity_potential_rep_trip_case124_pct"]
        + 0.15 * (measures["revisit_intent_mean_w_rep_trip_case124"] / 5 * 100)
    )
    rep_rows.append(
        {
            "spot_code": spot_code,
            "rep_trip_n_case124": len(subset),
            "rep_trip_weighted_denom_case124": weight.sum(),
            "is_reliable_rep_trip_case124": len(subset) >= 10,
            **measures,
        }
    )
representative_feature = pd.DataFrame(rep_rows)
feature = feature.merge(representative_feature, on="spot_code", how="left", validate="one_to_one")
feature["rep_trip_n_case124"] = feature["rep_trip_n_case124"].fillna(0).astype(int)
feature["rep_trip_weighted_denom_case124"] = feature["rep_trip_weighted_denom_case124"].fillna(0.0)
feature["is_reliable_rep_trip_case124"] = feature["is_reliable_rep_trip_case124"].fillna(False).astype(bool)

## 8. CASE 2 군집 구성과 기존 표 회귀검증

In [ ]:
cluster_source = rep_region.loc[rep_region["CASE"].eq(2)].merge(
    cluster_labels,
    on=["YEAR", "ID"],
    how="inner",
    validate="many_to_one",
)
assert set(cluster_source["군집"].dropna().unique()) == set(CLUSTER_LABELS)
cluster_wide = (
    cluster_source.pivot_table(index="spot_code", columns="군집", values="weighted_region_size", aggfunc="sum", fill_value=0.0)
    .reindex(columns=CLUSTER_LABELS, fill_value=0.0)
)
cluster_wide.columns = [f"군집{label}_가중규모" for label in CLUSTER_LABELS]
cluster_wide = cluster_wide.reset_index()
cluster_columns = [f"군집{label}_가중규모" for label in CLUSTER_LABELS]
cluster_wide["연고관광_가중규모_합"] = cluster_wide[cluster_columns].sum(axis=1)
for label in CLUSTER_LABELS:
    weighted_column = f"군집{label}_가중규모"
    share_column = f"군집{label}_구성비"
    cluster_wide[share_column] = np.where(
        cluster_wide["연고관광_가중규모_합"].gt(0),
        cluster_wide[weighted_column] / cluster_wide["연고관광_가중규모_합"],
        np.nan,
    )
assert np.allclose(
    cluster_wide.loc[cluster_wide["연고관광_가중규모_합"].gt(0), [f"군집{label}_구성비" for label in CLUSTER_LABELS]].sum(axis=1),
    1.0,
    atol=1e-6,
)
feature = feature.merge(cluster_wide, on="spot_code", how="left", validate="one_to_one")
feature[cluster_columns + ["연고관광_가중규모_합"]] = feature[
    cluster_columns + ["연고관광_가중규모_합"]
].fillna(0.0)

legacy_columns = [
    "시도명", "시군구명", "비가중_여행수", "가중_지역규모", "전국_대비_비중(%)",
    *CASE_LABELS.values(), *cluster_columns, "연고관광_가중규모_합",
    *[f"군집{label}_구성비" for label in CLUSTER_LABELS],
]
regression = feature.merge(
    legacy_feature[legacy_columns],
    on=["시도명", "시군구명"],
    suffixes=("_new", "_legacy"),
    validate="one_to_one",
)
regression_rows = []
for column in legacy_columns[2:]:
    new = regression[f"{column}_new"]
    old = regression[f"{column}_legacy"]
    max_difference = float(np.nanmax(np.abs(new.to_numpy() - old.to_numpy())))
    assert np.allclose(new, old, atol=1e-6, equal_nan=True), column
    regression_rows.append({"열": column, "최대_절대차": max_difference})
regression_qc = pd.DataFrame(regression_rows)
display(regression_qc)

## 9. 최종 코드북·품질검사·최소 QC 시각화

In [ ]:
feature = feature.reindex(columns=OUTPUT_COLUMNS).sort_values("spot_code").reset_index(drop=True)
assert len(feature) == 229
assert feature["spot_code"].is_unique
assert not feature[["시도명", "시군구명"]].duplicated().any()
assert np.isfinite(feature.select_dtypes(include=np.number).to_numpy()).all()
pct_columns = [column for column in feature if column.endswith("_pct") or column.endswith("(%)")]
assert feature[pct_columns].stack().between(0, 100).all()
assert feature["low_spending_conversion_hint_rep_trip_case124"].dropna().between(0, 100).all()
assert feature["satisfaction_mean_w_rep_trip_case124"].dropna().between(1, 5).all()
assert feature["revisit_intent_mean_w_rep_trip_case124"].dropna().between(1, 5).all()

group_by_column = {}
for column in OUTPUT_COLUMNS:
    if column in {"spot_code", "시도명", "시군구명"}:
        group_by_column[column] = "키"
    elif column.startswith("CASE") or column.startswith("case") or column.startswith("hometown") or column.startswith("rank_") or column in {"비가중_여행수", "가중_지역규모", "전국_대비_비중(%)"}:
        group_by_column[column] = "전체여행 규모·CASE 구성"
    elif column.startswith("군집") or column == "연고관광_가중규모_합":
        group_by_column[column] = "CASE 2 연고관광 군집"
    elif column.endswith("all_trip_case15") or column.endswith("all_trip_case15_pct"):
        group_by_column[column] = "CASE 1~5 공통 여행특성"
    else:
        group_by_column[column] = "대표여행 CASE 1·2·4 특성"
korean_label_by_column = {
    "spot_code": "방문지 시군구 코드",
    "시도명": "방문 시도명",
    "시군구명": "방문 시군구명",
    "비가중_여행수": "비가중 여행-시군구 건수",
    "가중_지역규모": "가중 시군구 방문 규모",
    "전국_대비_비중(%)": "전국 대비 가중 시군구 방문 비중",
    "hometown_visit_n_all_trip_case23": "연고방문 여행-시군구 비가중 건수",
    "hometown_visit_weighted_all_trip_case23": "연고방문 가중 시군구 방문 규모",
    "hometown_visit_share_w_all_trip_case15_pct": "전체여행 중 연고방문 가중 비중",
    "rank_by_hometown_visit_weighted_all_trip_case23": "연고방문 가중 방문 규모 순위",
    "rank_by_hometown_visit_share_all_trip_case23": "전체여행 중 연고방문 비중 순위",
    "연고관광_가중규모_합": "연고관광 군집 가중 규모 합계",
    "total_cost_mean_w_all_trip_case15": "총여행비용 가중 평균",
    "total_cost_median_w_all_trip_case15": "총여행비용 가중 중앙값",
    "one_person_cost_mean_w_all_trip_case15": "1인당 여행비용 가중 평균",
    "one_person_cost_median_w_all_trip_case15": "1인당 여행비용 가중 중앙값",
    "travel_nights_mean_w_all_trip_case15": "숙박일수 가중 평균",
    "overnight_rate_w_all_trip_case15_pct": "숙박여행 가중 비율",
    "family_friend_lodging_rate_w_all_trip_case15_pct": "가족·친지집 숙박 가중 비율",
    "rep_trip_n_case124": "대표여행-시군구 비가중 건수",
    "rep_trip_weighted_denom_case124": "대표여행 가중 분모",
    "family_friend_lodging_rate_w_rep_trip_case124_pct": "대표여행 가족·친지집 숙박 가중 비율",
    "no_reservation_rate_w_rep_trip_case124_pct": "대표여행 무예약 가중 비율",
    "nature_activity_rate_w_rep_trip_case124_pct": "대표여행 자연 활동 가중 비율",
    "food_activity_rate_w_rep_trip_case124_pct": "대표여행 음식 활동 가중 비율",
    "rest_activity_rate_w_rep_trip_case124_pct": "대표여행 휴식 활동 가중 비율",
    "family_visit_activity_rate_w_rep_trip_case124_pct": "대표여행 가족·친지 방문 활동 가중 비율",
    "satisfaction_mean_w_rep_trip_case124": "대표여행 만족도 가중 평균",
    "revisit_intent_mean_w_rep_trip_case124": "대표여행 재방문 의향 가중 평균",
    "activity_potential_rep_trip_case124_pct": "대표여행 활동 잠재력",
    "low_spending_conversion_hint_rep_trip_case124": "저지출 전환 탐색 점수",
    "is_reliable_rep_trip_case124": "대표여행 표본 신뢰 기준 충족 여부",
}
case_korean_names = {
    1: "관광·휴양",
    2: "관광 포함 연고방문",
    3: "관광 미포함 연고방문",
    4: "관광 포함 출장·업무",
    5: "관광 미포함 출장·업무",
}
for case, column in CASE_LABELS.items():
    korean_label_by_column[column] = f"{case_korean_names[case]} 가중 규모"
    korean_label_by_column[f"case{case}_trip_n_all_trip_case15"] = (
        f"{case_korean_names[case]} 여행-시군구 비가중 건수"
    )
    korean_label_by_column[f"case{case}_share_weighted_all_trip_case15_pct"] = (
        f"{case_korean_names[case]} 가중 구성비"
    )
for label in CLUSTER_LABELS:
    korean_label_by_column[f"군집{label}_가중규모"] = f"{label} 군집 가중 규모"
    korean_label_by_column[f"군집{label}_구성비"] = f"{label} 군집 구성비"
assert set(korean_label_by_column) == set(OUTPUT_COLUMNS)
codebook_rows = []
for column in OUTPUT_COLUMNS:
    group = group_by_column[column]
    if group == "대표여행 CASE 1·2·4 특성":
        scope = "CHECK=Y, CASE 1·2·4 대표여행이 방문한 시군구"
        weight = "WT_DOM, 여행-시군구 중복 제거 후 시군구별 중복 배정"
    elif group == "CASE 2 연고관광 군집":
        scope = "군집 라벨이 있는 대표 연고관광(CASE 2) 여행"
        weight = "WT_DOM × 시군구 체류일 비중"
    else:
        scope = "CASE 1~5 모든 유효 여행-시군구"
        weight = "WT_DOM × 시군구 체류일 비중"
    codebook_rows.append(
        {
            "column_name": column,
            "한글_표시명": korean_label_by_column[column],
            "열_그룹": group,
            "분석대상_분석단위": scope,
            "원천변수": "통합 all_tour, 지역코드, CASE 2 군집 라벨",
            "CASE_CHECK_조건": scope,
            "산식_가중치_분자_분모": weight,
            "결측_구조적비해당_처리": "변수별 결측은 분자·분모에서 함께 제외; 구조적 비해당은 모집단에서 제외",
            "단위_값범위": "비율은 %, 군집 구성비는 0~1, 만족·재방문은 1~5",
            "기존표_비교대상": column in legacy_columns,
            "포스터_권장표기": "대표여행 응답 특성" if group == "대표여행 CASE 1·2·4 특성" else "전체여행 특성",
            "주의사항_파생식": "정책점수는 탐색적 합성지표이며 인과효과나 모집단 비율로 해석하지 않음",
        }
    )
feature_codebook = pd.DataFrame(codebook_rows)
assert feature_codebook["column_name"].tolist() == feature.columns.tolist()

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(feature["rep_trip_n_case124"], bins=20, color=COLORS["primary"], edgecolor="white")
ax.axvline(10, color=COLORS["accent"], linestyle="--", label="신뢰 플래그 기준 N=10")
ax.set_xlabel("시군구별 대표여행 비가중 N")
ax.set_ylabel("시군구 수")
ax.set_title("대표여행 CASE 1·2·4 표본수 분포")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "rep_trip_n_distribution.png", dpi=300, bbox_inches="tight")
plt.show()
plt.close(fig)

## 10. 최종 저장과 요약

In [ ]:
FEATURE_DIR.mkdir(parents=True, exist_ok=True)
feature.to_csv(FEATURE_PATH, index=False, encoding="utf-8-sig")
feature_codebook.to_csv(FEATURE_CODEBOOK_PATH, index=False, encoding="utf-8-sig")
qc_visit.to_csv(TABLES_DIR / "visit_qc.csv", index=False, encoding="utf-8-sig")
unmatched_spot_codes.to_csv(TABLES_DIR / "unmatched_spot_codes.csv", index=False, encoding="utf-8-sig")
regression_qc.to_csv(TABLES_DIR / "legacy_regression_qc.csv", index=False, encoding="utf-8-sig")

print(f"최종 특성표: {FEATURE_PATH}")
print(f"최종 코드북: {FEATURE_CODEBOOK_PATH}")
print(f"행·열: {feature.shape[0]}행 × {feature.shape[1]}열")
print(f"대표여행 N 10 이상 시군구: {int(feature['is_reliable_rep_trip_case124'].sum())}개")
print("공통 특성은 CASE 1~5 전체여행, A 특성은 CHECK=Y CASE 1·2·4 대표여행 기준입니다.")

In [ ]:
from src.notebook_sync import sync_script_from_notebook

sync_script_from_notebook(
    "notebooks/05_Features_by_Region/260726_국민여행조사_시군구별_전체여행_특성_3개년.ipynb"
)